# Fase 4 (Parte B) — CNN desde Cero
**Proyecto:** FruitVision — Clasificación de Calidad de Frutas  
**Curso:** Algoritmos y Programación III — Semestre 2026-1  
**Integrantes:** Juan José Gordillo Córdoba · Sebastián Jiménez Galvis · Juan Pablo Zambrano Cortez

---

### Arquitectura Híbrida FruitVision

Este notebook entrena la componente de **deep learning** del sistema.

| Responsabilidad | Módulo | Entrada |
|----------------|--------|---------|
| Calidad (Premium / Estándar / Descarte) | **CNN** (este notebook) | Imagen 128×128 BGR |
| Tamaño (Pequeño / Mediano / Grande) | Función determinista OpenCV | `diameter_norm` de Fase 3 |

### Arquitectura CNN

```
Input(128, 128, 3)
  → Data Augmentation (solo en training)
  → [Conv2D(32) → BatchNorm → MaxPool] × 3  ← 3 bloques (patrón del curso, CIFAR-10)
  → GlobalAveragePooling2D                   ← reemplaza Flatten cuando los mapas son pequeños
  → Dense(128, relu)
  → Dropout(0.5)
  → Dense(3, softmax)                        ← 3 neuronas = 3 clases (regla del curso)
```

`sparse_categorical_crossentropy` porque las etiquetas son enteros 0/1/2 (no one-hot).  
`class_weight` calculado con `compute_class_weight('balanced')` para compensar IR ≈ 10.86.

## 0. Configuración del entorno

In [ ]:
import os
import pathlib
import warnings
import json

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # silenciar logs de TF compilación

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, Model
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── Resolución robusta de la raíz del proyecto ────────────────────────────────
ROOT = pathlib.Path.cwd()
for _ in range(6):
    if (ROOT / "data").exists():
        break
    ROOT = ROOT.parent

DATA_PROC   = ROOT / "data" / "processed" / "fase3"
MODELS_DIR  = ROOT / "models" / "saved"
FIGURES_DIR = ROOT / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Hiperparámetros ───────────────────────────────────────────────────────────
CNN_IMG_SIZE   = (128, 128)     # (H, W) de entrada a la CNN
BATCH_SIZE     = 32
EPOCHS         = 40
LEARNING_RATE  = 1e-3           # Adam lr inicial
DROPOUT_RATE   = 0.5
N_CLASSES      = 3
QUALITY_CLASSES = ["Premium", "Estándar", "Descarte"]
QUALITY_PALETTE = {
    "Premium" : "#2ecc71",
    "Estándar": "#f39c12",
    "Descarte": "#e74c3c",
}

print(f"TensorFlow  : {tf.__version__}")
print(f"Dispositivo : {tf.test.gpu_device_name() or 'CPU'}")
print(f"Raíz        : {ROOT}")
print(f"Tamaño img  : {CNN_IMG_SIZE}")

## 1. Carga de los manifests de Fase 3

In [ ]:
def load_manifest(split: str) -> pd.DataFrame:
    """
    Carga el manifest CSV de Fase 3 para el split indicado.
    FIX-CNN-1: autosuficiente, sin dependencias de src/.
    """
    csv_path = DATA_PROC / f"manifest_{split}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(
            f"No se encontró {csv_path}.\n"
            "Ejecutar primero el notebook de Fase 3."
        )
    df = pd.read_csv(csv_path)

    if "quality_id" not in df.columns and "quality" in df.columns:
        q2id = {"Premium": 0, "Estándar": 1, "Descarte": 2}
        df["quality_id"] = df["quality"].map(q2id)

    df = df[df["quality_id"].notna()].copy()
    df["quality_id"] = df["quality_id"].astype(int)
    print(f"[{split:5s}] {len(df):,} instancias  | "
          f"{dict(df['quality_id'].value_counts().sort_index())}")
    return df.reset_index(drop=True)


manifest_train = load_manifest("train")
manifest_val   = load_manifest("val")
manifest_test  = load_manifest("test")

## 2. Pipeline de datos con tf.data

> **FIX-CNN-3:** Carga lazy con `tf.data.Dataset` para no saturar la RAM.  
> Las imágenes se leen del disco en batches durante el entrenamiento.  
> El **data augmentation** se aplica SOLO en el dataset de train mediante
> `training=True`, que Keras gestiona automáticamente al llamar `model.fit()`.

In [ ]:
def path_to_tensor(path_str: str, label: int,
                  img_size: tuple = CNN_IMG_SIZE) -> tuple:
    """
    Lee una imagen BGR con OpenCV, la convierte a RGB float32 normalizado
    y la redimensiona a CNN_IMG_SIZE.

    Se usa como función numpy wrapeada en tf.data (tf.py_function),
    lo que permite usar OpenCV sin salir del ecosistema tf.data.
    """
    try:
        img_bgr = cv2.imread(str(path_str))
        if img_bgr is None:
            raise FileNotFoundError(str(path_str))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, img_size, interpolation=cv2.INTER_AREA)
        img_f32 = img_rgb.astype(np.float32) / 255.0
        return img_f32, int(label)
    except Exception:
        return np.zeros((*img_size, 3), dtype=np.float32), int(label)


def make_dataset(manifest: pd.DataFrame, split: str,
                 batch_size: int = BATCH_SIZE,
                 img_size: tuple = CNN_IMG_SIZE) -> tf.data.Dataset:
    """
    Construye un tf.data.Dataset a partir del manifest CSV.

    - train: shuffle + repeat + augmentation activa (training=True en fit)
    - val / test: sin shuffle, sin repeat
    """
    paths  = manifest["processed_path"].tolist()
    labels = manifest["quality_id"].tolist()

    def load_img(path, label):
        """Wrapper de tf.py_function para usar OpenCV dentro de tf.data."""
        img, lbl = tf.py_function(
            func=path_to_tensor,
            inp=[path, label],
            Tout=[tf.float32, tf.int32],
        )
        img.set_shape((*img_size, 3))
        lbl.set_shape(())
        return img, lbl

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if split == "train":
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(manifest_train, "train")
val_ds   = make_dataset(manifest_val,   "val")
test_ds  = make_dataset(manifest_test,  "test")

print("Datasets creados:")
print(f"  train_ds: {len(manifest_train):,} imágenes  |  {len(manifest_train)//BATCH_SIZE} batches/epoch")
print(f"  val_ds  : {len(manifest_val):,} imágenes")
print(f"  test_ds : {len(manifest_test):,} imágenes")

# Verificar un batch
sample_imgs, sample_labels = next(iter(train_ds))
print(f"\nBatch de muestra: imgs={sample_imgs.shape}  labels={sample_labels.shape}")
print(f"  Rango de valores: [{sample_imgs.numpy().min():.2f}, {sample_imgs.numpy().max():.2f}]")

### 2.1 Visualización de un batch de ejemplo

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
fig.suptitle("Muestra del batch de entrenamiento (antes de augmentación)",
             fontsize=13, fontweight="bold")

sample_imgs_np = sample_imgs.numpy()
sample_labels_np = sample_labels.numpy()

# Mostrar 8 imágenes por clase
shown = {0: 0, 1: 0, 2: 0}
row_map = {0: 0, 1: 1, 2: 2}
col_counters = [0, 0, 0]

for img, lbl in zip(sample_imgs_np, sample_labels_np):
    cls = int(lbl)
    row = row_map[cls]
    col = col_counters[cls]
    if col >= 8:
        continue
    axes[row, col].imshow(img)
    axes[row, col].axis("off")
    if col == 0:
        axes[row, col].set_ylabel(QUALITY_CLASSES[cls], fontsize=9,
                                   rotation=90, labelpad=30, va="center")
    col_counters[cls] += 1

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_cnn_batch_muestra.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Arquitectura CNN

### Diseño y justificaciones

La arquitectura sigue el patrón **CIFAR-10 del curso** (3 bloques Conv→BN→MaxPool),
adaptado al problema de calidad de frutas:

**¿Por qué BatchNormalization?**  
Normaliza las activaciones del mini-batch, acelerando el entrenamiento y
actuando como regularizador. Reduce la sensibilidad al learning rate inicial.

**¿Por qué GlobalAveragePooling en lugar de Flatten?**  
Con 3 MaxPool sobre 128×128, el mapa de activación llega a 16×16×128.
Flatten generaría 32,768 conexiones densas. GAP calcula la media espacial
de cada mapa → 128 valores, reduciendo drásticamente los parámetros y el overfitting.

**¿Por qué `padding='same'`?**  
Mantiene las dimensiones espaciales iguales tras la convolución (añade ceros en bordes).
Con `padding='valid'`, el mapa se reduce con cada conv, lo que requiere un diseño más cuidadoso.

**Número de filtros progresivo (32 → 64 → 128):**  
Las primeras capas aprenden features de bajo nivel (bordes, colores).
Las capas profundas combinan features para detectar patrones complejos (manchas, texturas de podredumbre).

In [ ]:
# FIX-CNN-2: Data augmentation como Sequential SEPARADO del modelo.
# Keras aplica estas capas solo durante training (training=True) de forma
# automática cuando están dentro del modelo. Al llamar model.predict()
# o model.evaluate(), training=False → no se augmenta.
# Esto es diferente a aplicar augmentación FUERA del modelo, pero es el
# patrón aceptado por el curso para mantener el modelo serializable.

data_aug_layer = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),           # ±10% de rotación
    layers.RandomZoom(0.10),               # ±10% de zoom
    layers.RandomBrightness(0.10, value_range=(0.0, 1.0)),
], name="data_augmentation")


def build_cnn(
    img_size     : tuple = CNN_IMG_SIZE,
    n_classes    : int   = N_CLASSES,
    dropout_rate : float = DROPOUT_RATE,
    lr           : float = LEARNING_RATE,
) -> models.Sequential:
    """
    Construye la CNN FruitVision desde cero.

    Arquitectura (patrón CIFAR-10 del curso adaptado a 3 clases):
    - 3 bloques Conv2D → BatchNorm → MaxPool2D
    - GlobalAveragePooling2D (reduce sobreajuste vs Flatten)
    - Dense(128, relu) → Dropout(0.5) → Dense(3, softmax)

    Función de pérdida:
    - sparse_categorical_crossentropy: etiquetas enteras 0/1/2 (no one-hot).
      Equivalente a categorical_crossentropy pero sin necesidad de convertir y.

    Métrica principal:
    - accuracy: suficiente para monitorear training; F1-macro se calcula en evaluación.
    """
    # API Funcional (patrón del profesor — módulo 11, CIFAR-10)
    inputs = tf.keras.Input(shape=(*img_size, 3), name="input_image")

    # Data augmentation (solo activa en training=True)
    x = data_aug_layer(inputs)

    # Bloque 1: 32 filtros → mapa: 64×64×32
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)

    # Bloque 2: 64 filtros → mapa: 32×32×64
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)

    # Bloque 3: 128 filtros → mapa: 16×16×128
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool3")(x)

    # Clasificador
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="dense1")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(n_classes, activation="softmax", name="output")(x)

    model = Model(inputs=inputs, outputs=outputs, name="FruitCNN_v1")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


cnn = build_cnn()
cnn.summary()

# Guardar resumen de arquitectura como texto
with open(MODELS_DIR / "cnn_architecture.txt", "w") as f:
    cnn.summary(print_fn=lambda x: f.write(x + "\n"))
print("\n[✓] Resumen de arquitectura guardado en models/saved/cnn_architecture.txt")

## 4. Entrenamiento

### Estrategia ante el desbalanceo

Con IR ≈ 10.86, la red tiende a predecir siempre Premium (clase mayoritaria).
`class_weight` penaliza los errores en la clase Estándar con mayor intensidad:

$$w_k = \frac{n}{K \cdot n_k}$$

donde $n$ = total de instancias, $K$ = número de clases, $n_k$ = instancias de clase $k$.

### Callbacks
- **EarlyStopping:** interrumpe el entrenamiento si `val_loss` no mejora en `patience=8` epochs, restaurando los mejores pesos.
- **ReduceLROnPlateau:** reduce `lr` en un factor de 0.5 si `val_loss` se estanca por `patience=4` epochs. Rango mínimo: `1e-6`.

In [ ]:
# class_weight calculado desde el train manifest (no desde el dataset de TF)
# para evitar iterar el dataset completo solo para contar clases.
y_train_arr  = manifest_train["quality_id"].values
classes_uniq = np.unique(y_train_arr)
cw_arr       = compute_class_weight("balanced", classes=classes_uniq, y=y_train_arr)
class_weight = dict(enumerate(cw_arr))

print("Pesos de clase para compensar el desbalanceo:")
for cid, cname in enumerate(QUALITY_CLASSES):
    n_k = (y_train_arr == cid).sum()
    print(f"  {cname:10s} (n_train={n_k:,}) → w = {class_weight[cid]:.4f}")

cbs = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    callbacks.ModelCheckpoint(
        filepath=str(ROOT / "models" / "checkpoints" / "cnn_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    ),
]

# Crear directorio de checkpoints
(ROOT / "models" / "checkpoints").mkdir(parents=True, exist_ok=True)

print(f"\nIniciando entrenamiento: {EPOCHS} epochs máx | batch={BATCH_SIZE}")
history = cnn.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = EPOCHS,
    class_weight    = class_weight,
    callbacks       = cbs,
    verbose         = 2,
)

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n✓ Entrenamiento finalizado")
print(f"  Epochs ejecutados : {len(history.history['val_loss'])}")
print(f"  Mejor epoch       : {best_epoch}")
print(f"  Mejor val_loss    : {min(history.history['val_loss']):.4f}")
print(f"  Mejor val_acc     : {max(history.history['val_accuracy']):.4f}")

## 5. Curvas de aprendizaje

Las curvas de pérdida y accuracy permiten diagnosticar:
- **Overfitting:** val_loss sube mientras train_loss baja → la red memoriza en lugar de generalizar.
- **Underfitting:** ambas curvas altas → el modelo no tiene suficiente capacidad.
- **Buen ajuste:** ambas curvas bajan juntas y convergen.

La línea vertical indica el epoch restaurado por `EarlyStopping`.

In [ ]:
h  = history.history
ep = list(range(1, len(h["loss"]) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Curvas de Aprendizaje — CNN FruitVision", fontsize=14, fontweight="bold")

for ax, (train_key, val_key, ylabel) in zip(axes, [
    ("loss",     "val_loss",     "Pérdida (sparse categorical crossentropy)"),
    ("accuracy", "val_accuracy", "Accuracy"),
]):
    ax.plot(ep, h[train_key], label="Train", color="#3498db", linewidth=2)
    ax.plot(ep, h[val_key],   label="Val",   color="#e74c3c", linewidth=2)
    ax.axvline(best_epoch, color="#2ecc71", ls="--", lw=1.8,
               label=f"Mejor epoch ({best_epoch})")
    ax.fill_between(ep,
                    h[train_key], h[val_key],
                    alpha=0.08, color="purple",
                    label="Brecha train–val")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9); ax.set_title(ylabel)

# FIX-CNN-5: anotar la brecha en el mejor epoch
best_val_loss   = min(h["val_loss"])
best_train_loss = h["loss"][best_epoch - 1]
brecha          = best_val_loss - best_train_loss
axes[0].annotate(
    f"Brecha = {brecha:.4f}",
    xy=(best_epoch, (best_val_loss + best_train_loss) / 2),
    xytext=(best_epoch + 1, best_val_loss),
    fontsize=8, color="purple",
    arrowprops={"arrowstyle": "->", "color": "purple"},
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_cnn_curvas.pdf", bbox_inches="tight")
plt.show()
print("[✓] Figura guardada → fase4_cnn_curvas.pdf")
print()
if brecha > 0.05:
    print(f"[!] Brecha train-val = {brecha:.4f} → posible overfitting moderado.")
    print("    Opciones: aumentar Dropout, reducir capacidad, más augmentación.")
else:
    print(f"[✓] Brecha train-val = {brecha:.4f} → generalización adecuada.")

## 6. Evaluación en el conjunto de test

In [ ]:
# FIX-CNN-6: extracción de predicciones y etiquetas desde tf.data.Dataset
y_true_list, y_pred_list = [], []

for batch_imgs, batch_labels in test_ds:
    probs = cnn(batch_imgs, training=False).numpy()   # training=False → no augmentation
    preds = probs.argmax(axis=1)
    y_pred_list.extend(preds.tolist())
    y_true_list.extend(batch_labels.numpy().tolist())

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)

acc_cnn = accuracy_score(y_true, y_pred)
f1_cnn  = f1_score(y_true, y_pred, average="macro")
f1_per  = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2])

print("═" * 65)
print("  CNN FruitVision — Evaluación en Test")
print("═" * 65)
print(f"  Accuracy       : {acc_cnn:.3f}")
print(f"  F1-macro       : {f1_cnn:.3f}   ← métrica principal")
print(f"  F1 Premium     : {f1_per[0]:.3f}")
print(f"  F1 Estándar    : {f1_per[1]:.3f}   ← clase crítica (minoritaria)")
print(f"  F1 Descarte    : {f1_per[2]:.3f}")
print()
print(classification_report(y_true, y_pred, target_names=QUALITY_CLASSES, digits=3))

### 6.1 Matriz de confusión

In [ ]:
fig, axx = plt.subplots(figsize=(6, 5.5))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=QUALITY_CLASSES).plot(
    ax=axx, cmap="Greens", colorbar=False
)
axx.set_title(
    f"CNN FruitVision — Matriz de Confusión (Test)\n"
    f"F1-macro = {f1_cnn:.3f}  |  Accuracy = {acc_cnn:.3f}",
    fontsize=11
)
axx.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_cnn_confusion.pdf", bbox_inches="tight")
plt.show()
print("[✓] Figura guardada → fase4_cnn_confusion.pdf")

# Análisis de errores
print("\nAnálisis de errores:")
for true_cls in range(N_CLASSES):
    for pred_cls in range(N_CLASSES):
        if true_cls != pred_cls and cm[true_cls, pred_cls] > 0:
            print(f"  {QUALITY_CLASSES[true_cls]:10s} → predicho como "
                  f"{QUALITY_CLASSES[pred_cls]:10s}: {cm[true_cls, pred_cls]} casos")

## 7. Visualización de Feature Maps (Mapas de Activación)

> **Patrón del curso (módulo 11):** visualizar las salidas de las capas convolucionales
> para interpretar qué features aprende la red.
>
> Los primeros filtros (capa conv1) suelen capturar bordes y colores.  
> Las capas profundas (conv3) capturan patrones más abstractos como manchas o texturas.

In [ ]:
# FIX-CNN-8: visualización de feature maps (patrón del curso módulo 11)
# Extraer un modelo que devuelve las salidas de las 3 capas convolucionales

layer_names = ["conv1", "conv2", "conv3"]
activation_model = Model(
    inputs  = cnn.input,
    outputs = [cnn.get_layer(name).output for name in layer_names],
    name    = "activation_extractor",
)

# Tomar una imagen de cada clase del test set para visualizar
samples_per_class = {0: None, 1: None, 2: None}
for batch_imgs, batch_labels in test_ds:
    for img, lbl in zip(batch_imgs.numpy(), batch_labels.numpy()):
        cls = int(lbl)
        if samples_per_class[cls] is None:
            samples_per_class[cls] = img
        if all(v is not None for v in samples_per_class.values()):
            break
    if all(v is not None for v in samples_per_class.values()):
        break

fig, axes = plt.subplots(N_CLASSES, len(layer_names) + 1, figsize=(18, 9))
fig.suptitle("Feature Maps — Primeros 8 filtros de cada bloque convolucional",
             fontsize=12, fontweight="bold")

for cls_idx, (cls_id, img_arr) in enumerate(samples_per_class.items()):
    if img_arr is None:
        continue
    input_tensor = img_arr[np.newaxis, ...]     # (1, H, W, 3)

    # Imagen original
    axes[cls_idx, 0].imshow(img_arr)
    axes[cls_idx, 0].set_title(
        f"Original\n{QUALITY_CLASSES[cls_id]}", fontsize=8, fontweight="bold",
        color=QUALITY_PALETTE[QUALITY_CLASSES[cls_id]]
    )
    axes[cls_idx, 0].axis("off")

    # Activaciones de cada capa
    activations = activation_model.predict(input_tensor, verbose=0)
    for col_idx, (act, lname) in enumerate(zip(activations, layer_names), start=1):
        # act tiene forma (1, H, W, n_filters) — promediamos los primeros 8 filtros
        n_filters_to_show = min(8, act.shape[-1])
        avg_map = act[0, :, :, :n_filters_to_show].mean(axis=-1)
        axes[cls_idx, col_idx].imshow(avg_map, cmap="viridis")
        axes[cls_idx, col_idx].set_title(f"{lname}\n(media {n_filters_to_show} filtros)",
                                          fontsize=8)
        axes[cls_idx, col_idx].axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_cnn_feature_maps.pdf", bbox_inches="tight")
plt.show()
print("[✓] Figura guardada → fase4_cnn_feature_maps.pdf")

## 8. Guardado del modelo y métricas

In [ ]:
# Guardar modelo en formato Keras nativo
model_path = MODELS_DIR / "cnn_quality.keras"
cnn.save(str(model_path))
print(f"[✓] Modelo guardado: {model_path}")

# FIX-CNN-7: guardar métricas CSV para Fase 5 (comparación global)
cnn_metrics = pd.DataFrame([{
    "modelo"       : "CNN (desde cero)",
    "accuracy"     : acc_cnn,
    "f1_macro"     : f1_cnn,
    "f1_Premium"   : f1_per[0],
    "f1_Estándar"  : f1_per[1],
    "f1_Descarte"  : f1_per[2],
    "best_epoch"   : best_epoch,
    "epochs_total" : len(history.history["val_loss"]),
    "img_size"     : str(CNN_IMG_SIZE),
    "batch_size"   : BATCH_SIZE,
    "lr_inicial"   : LEARNING_RATE,
    "dropout"      : DROPOUT_RATE,
}])
cnn_metrics.to_csv(MODELS_DIR / "cnn_metrics.csv", index=False)
print(f"[✓] Métricas CSV guardadas: models/saved/cnn_metrics.csv")

# Hiperparámetros y configuración
cnn_config = {
    "img_size"       : CNN_IMG_SIZE,
    "batch_size"     : BATCH_SIZE,
    "epochs_max"     : EPOCHS,
    "best_epoch"     : int(best_epoch),
    "lr_inicial"     : LEARNING_RATE,
    "dropout"        : DROPOUT_RATE,
    "class_weight"   : {QUALITY_CLASSES[k]: round(v, 4) for k, v in class_weight.items()},
    "val_loss_best"  : round(float(min(history.history["val_loss"])), 4),
    "val_acc_best"   : round(float(max(history.history["val_accuracy"])), 4),
    "test_accuracy"  : round(float(acc_cnn), 4),
    "test_f1_macro"  : round(float(f1_cnn), 4),
}
import json
with open(MODELS_DIR / "cnn_config.json", "w") as f:
    json.dump(cnn_config, f, indent=2)
print(f"[✓] Configuración JSON guardada: models/saved/cnn_config.json")

print("\nResumen final:")
print(f"  F1-macro (test)   : {f1_cnn:.3f}")
print(f"  Accuracy (test)   : {acc_cnn:.3f}")
print(f"  F1 Estándar (test): {f1_per[1]:.3f}   ← clase crítica")
print(f"  Mejor epoch       : {best_epoch} / {len(history.history['val_loss'])} ejecutados")

---
## Próximos pasos → Fase 5: Evaluación y comparación global

En el notebook de Fase 5 se cargará:
- `models/saved/ml_metrics.csv` (Random Forest + XGBoost)
- `models/saved/cnn_metrics.csv` (CNN)

para realizar la **comparación global** de todos los modelos, el análisis de errores entre clases,
la comparación con literatura y el análisis de impacto del sistema en el contexto del proyecto.

➡️ **Siguiente:** `5_evaluation/evaluation.ipynb`